# 🌲 Análise de Volume e Biometria de Eucalipto
---
Este notebook é um ambiente interativo para a consolidação e análise das métricas dendrométricas geradas pelo nosso pipeline de processamento LiDAR.

### Objetivos:
1. Analisar as saídas das redes neurais de segmentação semântica e de instâncias (FF3D e TreeIso).
2. Processar e validar as reconstruções topológicas de RayExtract a partir de QSMs (`.txt` e `.csv`).
3. Gerar tabelas e visualizações consolidadas de DAP (Diâmetro à Altura do Peito) e Volume Total/Comercial.

### Como usar:
* Altere a variável `PIPELINE_KIND` para alternar entre os modelos de predição via nuvem de pontos ou via estrutura de árvore (RayExtract).
* Altere `EVALUATION_KIND` para `'single'` (teste rápido de um método) ou `'all'` (benchmarking completo).

In [13]:
# =====================================================================
# 1. IMPORTAÇÕES E SETUP DE AMBIENTE
# =====================================================================
import re
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Configuração visual para os gráficos
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams['figure.figsize'] = (10, 6)

# Configuração do Path do Projeto
PROJECT_ROOT = Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importações dos módulos do pipeline
from process_commercial_volume import (
    DEFAULT_DBH_METHODS,
    DEFAULT_VOLUME_METHODS,
    compare_method_combinations,
)
from eucalipto.io import load_ply, save_ply_with_fields
from eucalipto.rayextract_processor import process_rayextract_file

print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


### 2. Configurações Globais e Parâmetros
Defina aqui qual pipeline você quer rodar e os parâmetros biológicos do inventário.

In [14]:
# =====================================================================
# 2. CONFIGURAÇÕES
# =====================================================================
# Opções: 'point_cloud' (FF3D/TreeISO) ou 'rayextract' (QSM)
PIPELINE_KIND = 'point_cloud'

# Opções: 'single' (Apenas os métodos abaixo) ou 'all' (Todos os métodos disponíveis)
EVALUATION_KIND = 'all'  
SINGLE_DBH_METHOD = 'ensemble'
SINGLE_VOLUME_METHOD = 'axis_profile'

# Constante Biológica
WOOD_DENSITY_KG_M3 = 900.0
POINT_CLOUD_CROP_FRACTION = 1.0

# ---------------------------------------------------------------------
# PARÂMETROS PARA 'point_cloud' (FF3D / TreeIso)
# ---------------------------------------------------------------------
POINT_CLOUD_PATH = Path('/home/matheuspimenta/euc/projeto_eucalipto/third_party/FF3D_inference/ff3d_forestsens/FF3D_oracle/bucket_out_folder/talhao_62_25x25_round2.ply')
POINT_CLOUD_TREE_ID_CANDIDATES = ['tree_id', 'instance_pred', 'treeID', 'final_segs']
POINT_CLOUD_TRUNK_CANDIDATES = ['trunk_leaf_label', 'semantic_seg', 'semantic_pred', 'leafwood_pred']
POINT_CLOUD_TRUNK_LABEL = 1

# ---------------------------------------------------------------------
# PARÂMETROS PARA 'rayextract'
# ---------------------------------------------------------------------
RAYEXTRACT_TREEFILES = {
    'plot_1': Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto/results/canoa/tests/62/rayextract_full/rayextract_work/diego_talhao_62_raycloud_trees.txt'),
    # 'plot_2': Path('/home/matheuspimenta/Jobs/Eucalipto/drive/outputs/ray/CULS_CULS_plot_2_annotated_test_raycloud_trees.txt'),
    # 'plot_3': Path('/home/matheuspimenta/Jobs/Eucalipto/drive/outputs/ray/CULS_CULS_plot_3_annotated_val_raycloud_trees.txt'),
}

# Meshes da reconstrução 3D do RayExtract para comparação com o .txt.
# Informe uma lista de arquivos .ply por plot (ex.: tree_mesh.ply por árvore).
RAYEXTRACT_MESH_FILES = {
    'plot_1': [
        Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto/results/canoa/tests/62/rayextract_full/rayextract_work/diego_talhao_62_raycloud_trees_mesh.ply'),
    #     Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto/results/.../tree_001_mesh.ply'),
    ],
}

print(f"Pipeline Selecionado: {PIPELINE_KIND.upper()}")
print(f"Modo de Avaliação: {EVALUATION_KIND.upper()}")

Pipeline Selecionado: POINT_CLOUD
Modo de Avaliação: ALL


In [15]:
# =====================================================================
# 3. FUNÇÃO CORE DE PROCESSAMENTO (Point Cloud)
# =====================================================================
def crop_point_cloud_lower_left_quadrant(cloud_path, output_dir, crop_fraction=POINT_CLOUD_CROP_FRACTION):
    """Recorta a nuvem no quadrante inferior esquerdo do bounding box."""
    from eucalipto.io import load_cloud_auto, save_laz, save_ply_with_fields
    
    points, extras, format_label = load_cloud_auto(str(cloud_path))
    if points.size == 0:
        raise ValueError(f'Point cloud vazia: {cloud_path}')

    x_min, y_min = points[:, 0].min(), points[:, 1].min()
    x_max, y_max = points[:, 0].max(), points[:, 1].max()
    x_cut = x_min + (x_max - x_min) * float(crop_fraction)
    y_cut = y_min + (y_max - y_min) * float(crop_fraction)

    crop_mask = (points[:, 0] <= x_cut) & (points[:, 1] <= y_cut)
    cropped_points = points[crop_mask]
    cropped_extras = {name: values[crop_mask] for name, values in extras.items()}

    if cropped_points.size == 0:
        raise ValueError(
            f'Cropping removed all points from {Path(cloud_path).name}. Check crop_fraction or cloud extent.'
        )

    # Output format matches the input format (.laz, .las or .ply)
    ext = Path(cloud_path).suffix.lower()
    cropped_path = Path(output_dir) / f'{Path(cloud_path).stem}_crop_{int(crop_fraction * 100):02d}{ext}'
    
    if ext in {'.laz', '.las'}:
        save_laz(str(cropped_path), cropped_points, cropped_extras, like_path=str(cloud_path))
    else:
        save_ply_with_fields(str(cropped_path), cropped_points, cropped_extras)
        
    return cropped_path, int(cropped_points.shape[0]), int(points.shape[0])

def run_point_cloud_evaluation(
    cloud_path,
    dbh_methods,
    volume_methods,
    trunk_label=POINT_CLOUD_TRUNK_LABEL,
    wood_density_kg_m3=WOOD_DENSITY_KG_M3,
    tree_id_candidates=None,
    trunk_candidates=None,
    save_mesh_dir=None,
):
    """Encapsula a função de extração de métricas de nuvens de pontos segmentadas."""
    tree_id_candidates = tree_id_candidates or POINT_CLOUD_TREE_ID_CANDIDATES
    trunk_candidates = trunk_candidates or POINT_CLOUD_TRUNK_CANDIDATES
    dbh_df, combo_df = compare_method_combinations(
        str(cloud_path),
        tree_id_candidates=tree_id_candidates,
        trunk_leaf_candidates=trunk_candidates,
        trunk_label=trunk_label,
        wood_density_kg_m3=wood_density_kg_m3,
        dbh_methods=dbh_methods,
        volume_methods_list=volume_methods,
        save_mesh_dir=save_mesh_dir,
    )
    if not dbh_df.empty and 'tree_id' in dbh_df.columns:
        dbh_df = dbh_df[dbh_df['tree_id'].fillna(-1).astype(int) >= 0].copy()
    if not combo_df.empty and 'tree_id' in combo_df.columns:
        combo_df = combo_df[combo_df['tree_id'].fillna(-1).astype(int) >= 0].copy()
    return dbh_df, combo_df


In [16]:
# =====================================================================
# 4. EXECUÇÃO DO PIPELINE: NUVEM DE PONTOS (FF3D / TreeIso)
# =====================================================================
if PIPELINE_KIND == 'point_cloud':
    if not POINT_CLOUD_PATH.exists():
        raise FileNotFoundError(f'ERRO: Arquivo de nuvem não encontrado em: {POINT_CLOUD_PATH}')

    print(f"Iniciando processamento da nuvem: {POINT_CLOUD_PATH.name}...")
    
    # Define a lista de métodos baseada no tipo de avaliação escolhida
    if EVALUATION_KIND == 'single':
        selected_dbh_methods = [SINGLE_DBH_METHOD]
        selected_volume_methods = [SINGLE_VOLUME_METHOD]
        print(f"Rodando análise rápida -> DAP: '{SINGLE_DBH_METHOD}' | Vol: '{SINGLE_VOLUME_METHOD}'")
        
    elif EVALUATION_KIND == 'all':
        selected_dbh_methods = DEFAULT_DBH_METHODS
        selected_volume_methods = DEFAULT_VOLUME_METHODS
        print(f"🌎 Rodando benchmarking completo para todos os métodos.")
    else:
        raise ValueError("ERRO: EVALUATION_KIND deve ser 'single' ou 'all'")

    # Recorta a nuvem antes da análise para reduzir o custo computacional
    with tempfile.TemporaryDirectory(prefix='eucalipto_crop_') as crop_dir:
        cropped_cloud_path, cropped_n_points, original_n_points = crop_point_cloud_lower_left_quadrant(
            POINT_CLOUD_PATH,
            crop_dir,
            crop_fraction=POINT_CLOUD_CROP_FRACTION,
        )
        print(
            f"Nuvem recortada para {cropped_n_points}/{original_n_points} pontos ",
            f"({cropped_n_points / original_n_points:.1%}) no quadrante inferior esquerdo."
        )

        # Executa a função core e obtém os DataFrames
        dbh_df, volume_df = run_point_cloud_evaluation(
            cropped_cloud_path,
            dbh_methods=selected_dbh_methods,
            volume_methods=selected_volume_methods,
            save_mesh_dir='results/plot_meshes',
        )
    print("Processamento concluído com sucesso!")


Iniciando processamento da nuvem: talhao_62_25x25_round2.ply...
🌎 Rodando benchmarking completo para todos os métodos.
Nuvem recortada para 345712/345712 pontos  (100.0%) no quadrante inferior esquerdo.
✓ Saved merged tree meshes to results/plot_meshes/talhao_62_25x25_round2_crop_100_mesh_ensemble_cylinder.ply
✓ Saved merged tree meshes to results/plot_meshes/talhao_62_25x25_round2_crop_100_mesh_ensemble_voxel.ply
✓ Saved merged tree meshes to results/plot_meshes/talhao_62_25x25_round2_crop_100_mesh_ensemble_taper.ply
✓ Saved merged tree meshes to results/plot_meshes/talhao_62_25x25_round2_crop_100_mesh_ensemble_frustum.ply
✓ Saved merged tree meshes to results/plot_meshes/talhao_62_25x25_round2_crop_100_mesh_ensemble_axis_profile.ply
✓ Saved merged tree meshes to results/plot_meshes/talhao_62_25x25_round2_crop_100_mesh_single_ransac_cylinder.ply
✓ Saved merged tree meshes to results/plot_meshes/talhao_62_25x25_round2_crop_100_mesh_single_ransac_voxel.ply
✓ Saved merged tree meshes to 

### Resultados da Extração Direta da Nuvem (Point Cloud)
Os DataFrames abaixo consolidam os resultados do *fitting* por indivíduo arbóreo e o sumário estatístico do método de volumetria escolhido.

In [17]:
# =====================================================================
# 5. VISUALIZAÇÃO DE DADOS (Point Cloud)
# =====================================================================
if PIPELINE_KIND == 'point_cloud':
    
    # 5.1. Resultados Individuais (Árvore a Árvore)
    display(Markdown("#### **Resultados de DAP por Árvore**"))
    display(dbh_df.sort_values(['tree_id', 'method']).reset_index(drop=True))

    display(Markdown("#### **Resultados Volumétricos por Árvore e Combinação de Método**"))
    display(volume_df.sort_values(['tree_id', 'dbh_method', 'volume_method']).reset_index(drop=True))

    # 5.2. Sumário Estatístico (Apenas se o volume_df não estiver vazio)
    if not volume_df.empty:
        display(Markdown("#### **Sumário Global por Estratégia de Volume**"))
        summary = (
            volume_df.groupby('volume_method', dropna=False)
            .agg(
                Qtd_Árvores=('tree_id', 'nunique'),
                Volume_Médio_m3=('volume_m3', 'mean'),
                Volume_Mediano_m3=('volume_m3', 'median'),
                Vol_Comercial_Médio_m3=('commercial_volume_m3', 'mean'),
                Massa_Média_kg=('mass_kg', 'mean'),
            )
            .reset_index()
            .sort_values('volume_method')
        )
        # Exibe com fundo gradiente para facilitar a comparação
        display(summary.style.background_gradient(subset=['Volume_Médio_m3'], cmap='viridis'))

        if EVALUATION_KIND == 'all':
            display(Markdown("#### **Comparativo Cruzado: Método de DAP x Método de Volume**"))
            comparison = (
                volume_df.groupby(['dbh_method', 'volume_method'], dropna=False)
                .agg(
                    Qtd_Árvores=('tree_id', 'nunique'),
                    Volume_Médio_m3=('volume_m3', 'mean'),
                    Vol_Comercial_Médio_m3=('commercial_volume_m3', 'mean'),
                    Massa_Média_kg=('mass_kg', 'mean'),
                )
                .reset_index()
                .sort_values(['dbh_method', 'volume_method'])
            )
            display(comparison.style.background_gradient(subset=['Volume_Médio_m3'], cmap='plasma'))

#### **Resultados de DAP por Árvore**

,cloud,tree_id,method,dbh_cm,status,n_values
0,talhao_62_25x25_round2_crop_100,0,ensemble,NaN,none,0
1,talhao_62_25x25_round2_crop_100,0,ls,20.356873,ok,0
2,talhao_62_25x25_round2_crop_100,0,single_ransac,NaN,none,0
3,talhao_62_25x25_round2_crop_100,1,ensemble,NaN,none,0
4,talhao_62_25x25_round2_crop_100,1,ls,NaN,none,0
...,...,...,...,...,...,...
118,talhao_62_25x25_round2_crop_100,80,ls,NaN,none,0
119,talhao_62_25x25_round2_crop_100,80,single_ransac,NaN,none,0
120,talhao_62_25x25_round2_crop_100,94,ensemble,NaN,none,0
121,talhao_62_25x25_round2_crop_100,94,ls,NaN,none,0


#### **Resultados Volumétricos por Árvore e Combinação de Método**

,cloud,tree_id,dbh_method,volume_method,dbh_cm,height_m,cbc_rel_m,cbc_abs_m,volume_m3,mass_kg,commercial_volume_m3,commercial_mass_kg,status,n_trunk_pts,n_commercial_pts
0,talhao_62_25x25_round2_crop_100,0,ensemble,axis_profile,NaN,1.704500,1.414735,3.708235,0.526021,4.734190e+02,0.734743,661.269140,ok,86,84
1,talhao_62_25x25_round2_crop_100,0,ensemble,cylinder,NaN,1.704500,1.414735,3.708235,NaN,NaN,NaN,NaN,error: dbh_cm or CBC is missing for cylinder v...,86,84
2,talhao_62_25x25_round2_crop_100,0,ensemble,frustum,NaN,1.704500,1.414735,3.708235,2.105229,1.894706e+03,1.406592,1265.932909,ok,86,84
3,talhao_62_25x25_round2_crop_100,0,ensemble,taper,NaN,1.704500,1.414735,3.708235,27738.487836,2.496464e+07,7.780162,7002.145995,ok,86,84
4,talhao_62_25x25_round2_crop_100,0,ensemble,voxel,NaN,1.704500,1.414735,3.708235,0.010625,9.562500e+00,0.010375,9.337500,ok,86,84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
610,talhao_62_25x25_round2_crop_100,94,single_ransac,axis_profile,NaN,17.139301,3.427860,5.583260,NaN,NaN,NaN,NaN,error: Insufficient points for 'axis_profile' ...,5,2
611,talhao_62_25x25_round2_crop_100,94,single_ransac,cylinder,NaN,17.139301,3.427860,5.583260,NaN,NaN,NaN,NaN,error: dbh_cm or CBC is missing for cylinder v...,5,2
612,talhao_62_25x25_round2_crop_100,94,single_ransac,frustum,NaN,17.139301,3.427860,5.583260,NaN,NaN,NaN,NaN,error: Insuficientes amostras de raio para o m...,5,2
613,talhao_62_25x25_round2_crop_100,94,single_ransac,taper,NaN,17.139301,3.427860,5.583260,NaN,NaN,NaN,NaN,error: Insuficientes amostras de raio para o m...,5,2


#### **Sumário Global por Estratégia de Volume**

,volume_method,Qtd_Árvores,Volume_Médio_m3,Volume_Mediano_m3,Vol_Comercial_Médio_m3,Massa_Média_kg
0,axis_profile,41,6.334989,3.355794,5.667684,5701.490340
1,cylinder,41,5.243818,2.986525,4.028652,4719.436109
2,frustum,41,5.718832,2.844192,5.091149,5146.948993
3,taper,41,2870.078706,12.706656,1099.834713,2583070.834987
4,voxel,41,0.035622,0.029438,0.032353,32.059688


#### **Comparativo Cruzado: Método de DAP x Método de Volume**

,dbh_method,volume_method,Qtd_Árvores,Volume_Médio_m3,Vol_Comercial_Médio_m3,Massa_Média_kg
0,ensemble,axis_profile,41,6.334989,5.667684,5701.490340
1,ensemble,cylinder,41,6.045122,4.018810,5440.610092
2,ensemble,frustum,41,4.497406,6.355950,4047.665418
3,ensemble,taper,41,3015.060696,2141.725421,2713554.626476
4,ensemble,voxel,41,0.035622,0.032353,32.059688
5,ls,axis_profile,41,6.334989,5.667684,5701.490340
6,ls,cylinder,41,4.041188,3.376637,3637.069276
7,ls,frustum,41,6.502560,5.506435,5852.304417
8,ls,taper,41,3166.236891,527.737403,2849613.202226
9,ls,voxel,41,0.035622,0.032353,32.059688


In [18]:
volume_df.dropna(subset=['volume_m3'], inplace=False).shape

(264, 15)

In [19]:
dbh_df.to_csv('dbh_results.csv', index=False)
volume_df.to_csv('volume_results.csv', index=False)

In [20]:
# =====================================================================
# 6. EXECUÇÃO DO PIPELINE: RAYEXTRACT (QSMs + Meshes - Otimizado)
# =====================================================================
import re
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

if PIPELINE_KIND == 'rayextract':
    def _infer_tree_id_from_name(mesh_path: Path, fallback_id: int) -> int:
        matches = re.findall(r'\d+', mesh_path.stem)
        return int(matches[-1]) if matches else int(fallback_id)

    def process_single_mesh(mesh_path, wood_density_kg_m3, fallback_id):
        """Processa um único arquivo de malha de forma isolada (Ideal para paralelismo)"""
        import trimesh
        source_tree_hint = _infer_tree_id_from_name(mesh_path, fallback_id)
        
        try:
            # force='mesh' garante que seja um único objeto, ignorando hierarquias visuais da cena
            mesh = trimesh.load(str(mesh_path), force='mesh')
            
            # --- OTIMIZAÇÃO DE PERFORMANCE AQUI ---
            # Em vez de mesh.split(), calculamos as métricas globais da malha consolidada.
            z_min = float(mesh.bounds[0][2])
            z_max = float(mesh.bounds[1][2])
            height_m = float(max(0.0, z_max - z_min))

            # Tenta pegar o volume direto da malha. Se a malha estiver 'furada', ele retorna 0 ou erro.
            volume_source = 'mesh_global'
            volume_m3 = float(abs(mesh.volume)) if mesh.volume is not None else np.nan
            
            # Fallback rápido: Convex Hull da nuvem de vértices
            if not np.isfinite(volume_m3) or volume_m3 <= 0:
                hull = mesh.convex_hull
                volume_m3 = float(abs(hull.volume))
                volume_source = 'convex_hull_global'

            mass_kg = float(volume_m3 * wood_density_kg_m3) if np.isfinite(volume_m3) else np.nan

            return {
                'tree_id': fallback_id,
                'source_tree_hint': source_tree_hint,
                'mesh_file': mesh_path.name,
                'mesh_component_id': 0, # Tratado como componente único
                'component_count': 1,
                'volume_m3': volume_m3,
                'mass_kg': mass_kg,
                'height_m': height_m,
                'volume_source': volume_source,
                'status': 'success',
            }
        except Exception as exc:
            return {
                'tree_id': fallback_id,
                'source_tree_hint': source_tree_hint,
                'mesh_file': mesh_path.name,
                'mesh_component_id': None,
                'component_count': np.nan,
                'volume_m3': np.nan,
                'mass_kg': np.nan,
                'height_m': np.nan,
                'volume_source': 'error',
                'status': f'error: {exc}',
            }

    def process_rayextract_mesh_files_fast(mesh_paths, wood_density_kg_m3=WOOD_DENSITY_KG_M3):
        """Gerencia a extração usando pool de processos para não travar o notebook."""
        try:
            import trimesh
        except ImportError:
            print("⚠️ [AVISO] Pacote 'trimesh' não instalado. Pulando análise por mesh (.ply).")
            return pd.DataFrame()

        rows = []
        valid_paths = sorted({Path(p) for p in mesh_paths if Path(p).exists()})
        
        print(f"⏳ Processando {len(valid_paths)} meshes de forma otimizada...")
        start_time = time.time()
        
        # Uso de paralelismo para acelerar leitura de I/O pesado
        with ProcessPoolExecutor() as executor:
            futures = [
                executor.submit(process_single_mesh, path, wood_density_kg_m3, idx)
                for idx, path in enumerate(valid_paths)
            ]
            for future in as_completed(futures):
                rows.append(future.result())

        elapsed = time.time() - start_time
        print(f"⚡ Extração de Meshes concluída em {elapsed:.2f} segundos!")
        
        out = pd.DataFrame(rows)
        if not out.empty:
            out = out.sort_values(['mesh_file']).reset_index(drop=True)
        return out

    # =========================================================
    # Lógica Principal do Pipeline
    # =========================================================
    ray_txt_results = {}
    ray_mesh_results = {}

    print("🚀 Iniciando pipeline RayExtract (.txt vs .ply)...")

    for plot_name, txt_path in RAYEXTRACT_TREEFILES.items():
        print(f"\n================== {plot_name.upper()} ==================")

        # A) Estrutura cilíndrica (.txt)
        if txt_path.exists():
            print(f"📄 [TXT] Processando: {txt_path.name}")
            df_ray_txt = process_rayextract_file(str(txt_path), wood_density_kg_m3=WOOD_DENSITY_KG_M3)
            df_ray_txt.insert(0, 'plot', plot_name)
            ray_txt_results[plot_name] = df_ray_txt
        else:
            print(f"⚠️ [TXT] Arquivo não encontrado: {txt_path}")

        # B) Reconstrução geométrica (.ply) otimizada
        mesh_files_cfg = RAYEXTRACT_MESH_CSVS.get(plot_name, []) # Correção da variável que chama os meshes
        mesh_files = [Path(p) for p in mesh_files_cfg] if isinstance(mesh_files_cfg, list) else []

        if mesh_files:
            df_ray_mesh = process_rayextract_mesh_files_fast(mesh_files, wood_density_kg_m3=WOOD_DENSITY_KG_M3)
            if not df_ray_mesh.empty:
                df_ray_mesh.insert(0, 'plot', plot_name)
                ray_mesh_results[plot_name] = df_ray_mesh
        else:
            print(f"⚠️ [MESH] Nenhum arquivo .ply configurado para {plot_name}.")

    # =========================================================
    # CONSOLIDAÇÃO VISUAL
    # =========================================================
    ray_txt_all = pd.concat(ray_txt_results.values(), ignore_index=True) if ray_txt_results else pd.DataFrame()
    ray_mesh_all = pd.concat(ray_mesh_results.values(), ignore_index=True) if ray_mesh_results else pd.DataFrame()
    
    if (not ray_txt_all.empty) and (not ray_mesh_all.empty):
        txt_cmp = ray_txt_all.groupby('plot', dropna=False).agg(
            txt_vol_m3=('volume_m3', 'sum'), txt_qtd=('tree_id', 'nunique')).reset_index()

        mesh_cmp = ray_mesh_all.groupby('plot', dropna=False).agg(
            mesh_vol_m3=('volume_m3', 'sum'), mesh_qtd=('tree_id', 'nunique')).reset_index()

        comparison = txt_cmp.merge(mesh_cmp, on='plot', how='outer')
        comparison['Erro_Relativo_%'] = ((comparison['mesh_vol_m3'] - comparison['txt_vol_m3']) / comparison['txt_vol_m3']) * 100.0

        display(Markdown("#### **📊 Análise de Discrepância: Estrutura (.txt) vs Malha 3D (.ply)**"))
        display(comparison.style.background_gradient(subset=['Erro_Relativo_%'], cmap='coolwarm').format(precision=2))